<a href="https://colab.research.google.com/github/a-forty-two/EY-MarApr26/blob/main/07_OpenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab: Generative AI with Open AI

## Overview

This lab introduces **Generative AI** using the **Azure OpenAI Service** - Microsoft's managed deployment of OpenAI's models on Azure infrastructure. You will learn how to call a large language model (LLM) via the OpenAI Python SDK, build a multi-turn conversation with persistent chat history.

## What You Will Learn

By the end of this lab, you will be able to:

- Connect to an Azure OpenAI endpoint using the Python SDK
- Send a chat completion request with a system prompt, user messages, and assistant context
- Extract and display generated text from the API response
- Build a persistent chat history array that maintains context across multiple conversation turns
- Understand how the `messages` array drives multi-turn dialogue

## Prerequisites

- Azure OpenAI resource with at least two deployments:
  - A chat model (e.g. `gpt-4o`)
- API key and endpoint provided by your instructor
- Python packages: `openai`, `requests`

## Chat Completions with Azure OpenAI

## Step 1: Import the Azure OpenAI SDK

Import the `AzureOpenAI` client class from the `openai` package. This is the entry point for all chat and completion requests to your Azure-hosted model.

In [1]:
!pip install langchain langchain-openai langchain-community langchain-classic langgraph pillow python-dotenv pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
# Add Azure OpenAI package
from openai import AzureOpenAI

**Please get the key, endpoint and deployment name from your instructor"**

## Step 2: Set Your Azure OpenAI Credentials

Enter the endpoint URL, API key, and deployment name provided by your instructor. These three values are required to authenticate and route your request to the correct model deployment.

In [3]:
azure_oai_endpoint = "https://agentic-labs-v2.openai.azure.com/"
azure_oai_key = ""
azure_oai_deployment = "gpt-4o"
azure_api_version = "2024-12-01-preview"

## Step 3: Initialise the Azure OpenAI Client

Create an `AzureOpenAI` client object using your credentials. The `api_version` parameter specifies which version of the Azure OpenAI REST API to use — this must match a version supported by your resource.

In [4]:
# Initialize the Azure OpenAI client
client = AzureOpenAI(azure_endpoint = azure_oai_endpoint,api_key=azure_oai_key, api_version=azure_api_version)

## Step 4: Send a Chat Completion Request

Send a request to the model using `client.chat.completions.create()`. The `messages` array defines the full conversation context passed to the model:

- `system` - sets the model's persona and behaviour
- `user` - represents the human's message
- `assistant` - provides prior model responses as context

The model uses all messages in sequence to generate a contextually aware response. `temperature` controls randomness (0 = deterministic, 1 = creative) and `max_tokens` caps the response length.

In [5]:
# Add code to send request...
# Send request to Azure OpenAI model
response = client.chat.completions.create(
    model=azure_oai_deployment,
    temperature=0.7,
    max_tokens=400,
    messages=[
        {"role": "system", "content": "You are a helpful assistant with a lot of humor."},
        {"role": "user", "content": "Do penguins like pizzas?"},
        {"role": "assistant", "content": "Yes, penguins like pizza very much. They also like chips and hummus."},
        {"role": "user", "content": "Do polar bears support pizzas too?"}
    ]
)


print(response)

ChatCompletion(id='chatcmpl-DRXmqqkhv5u5SNRLMsPx0AnszKwsQ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Oh, absolutely! Polar bears are *huge* fans of pizza. Their favorite topping? Seal meat. Though I hear they’re trying to cut back for health reasons—too much ice cream dessert afterward, you know. But don’t try delivering a pizza to the Arctic; the delivery guy might not make it back. 🐾🍕', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'filtered': False, 'detected': False}, 'protected_material_text': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})], created=1775457104, model='gpt-4o-2024-11-20', object='chat.completion', service_t

## Step 5: Extract and Display the Generated Text

The raw API response contains metadata alongside the generated content. Extract just the text using `response.choices[0].message.content` and print it for readability.

In [6]:
generated_text = response.choices[0].message.content

# Print the response
print("Response: " + generated_text + "\n")

Response: Oh, absolutely! Polar bears are *huge* fans of pizza. Their favorite topping? Seal meat. Though I hear they’re trying to cut back for health reasons—too much ice cream dessert afterward, you know. But don’t try delivering a pizza to the Arctic; the delivery guy might not make it back. 🐾🍕



**Maintain Chat History**

## Step 6: Initialise the Chat History Array

To maintain context across multiple turns, store all messages — system, user, and assistant — in a persistent `messages_array`. Each new interaction is appended to this array before being sent to the model, so the model always sees the full conversation history.

In [7]:
# To maintain chat hisotry, we initialize messages array and append all interactions to the messages array
messages_array = [ {"role": "system", "content": "You are a helpful assistant with a lot of humor."},
        {"role": "user", "content": "Do penguins like pizzas?"},
        {"role": "assistant", "content": "Yes, penguins like pizza very much. They also like chips and hummus."}

        ]

## Step 7: Define the Next User Message

Set the next user message as a string variable. In a production chatbot, this would come from a user input field or API request.

In [8]:
input_text = "Do polar bears support pizzas too?"

## Step 8: Append the User Message and Send the Request

Append the new user message to the history array, then send the entire array to the model. This ensures the model has full context. After receiving the response, append the generated text back to the array as an `assistant` message — ready for the next turn.

In [9]:
# Add code to send request...
# Send request to Azure OpenAI model
messages_array.append({"role": "user", "content": input_text})

response = client.chat.completions.create(
    model=azure_oai_deployment,
    temperature=0.7,
    max_tokens=1200,
    messages=messages_array
)
generated_text = response.choices[0].message.content
# Add generated text to messages array
messages_array.append({"role": "system", "content": generated_text})

# Print generated text
print("Summary: " + generated_text + "\n")

Summary: Absolutely! Polar bears are big fans of *deep-dish* pizzas—because, you know, they like their meals hearty to survive those chilly Arctic days. Their favorite topping? Seal sausage... kidding! Polar bears don't actually eat pizza (or seals with toppings), but if they did, they’d probably insist it comes with extra ice on the side. Classic polar bear move! 🐾🍕



## Step 9: Inspect the Full Messages Array

Print the complete `messages_array` to see how the conversation history has grown across all turns. This is the exact payload that would be sent on the next API call — demonstrating how multi-turn memory works in a chat application.

In [10]:
print(messages_array)

[{'role': 'system', 'content': 'You are a helpful assistant with a lot of humor.'}, {'role': 'user', 'content': 'Do penguins like pizzas?'}, {'role': 'assistant', 'content': 'Yes, penguins like pizza very much. They also like chips and hummus.'}, {'role': 'user', 'content': 'Do polar bears support pizzas too?'}, {'role': 'system', 'content': "Absolutely! Polar bears are big fans of *deep-dish* pizzas—because, you know, they like their meals hearty to survive those chilly Arctic days. Their favorite topping? Seal sausage... kidding! Polar bears don't actually eat pizza (or seals with toppings), but if they did, they’d probably insist it comes with extra ice on the side. Classic polar bear move! 🐾🍕"}]


## Lab Summary

In this lab you explored two core capabilities of Generative AI:

**Chat Completions**

1. Connected to Azure OpenAI using the Python SDK with endpoint, key, and deployment credentials.
2. Sent a structured `messages` array containing system persona, prior user messages, and assistant responses.
3. Extracted generated text from the API response object.
4. Built a persistent `messages_array` that accumulates conversation history across turns, enabling true multi-turn dialogue where the model retains context.